# WFCRL IPPO and MAPPO on Ablaincourt

This notebook calls the authors' `wfcrl-benchmark/algos/baseline_ippo.py` and `baseline_mappo.py`. The separate benchmark checkout has two compatibility fixes: `action.detach()` in the deterministic evaluation action and scalar extraction of the common reward under NumPy 2. Neither changes PPO updates. Both write TensorBoard events in `wfcrl-benchmark/runs`. The default below uses the Wind-MARL comparison reward (`load_coef=0.1`); the paper scripts default to 1. Set it back to 1 when reproducing their published setting.

Select the Python kernel from `Wind-MARL/.venv` before running.

In [ ]:
from pathlib import Path
import os, subprocess, sys

ROOT = Path.cwd().resolve()
ENV_REPO = ROOT / "wfcrl-env"
BENCH = ROOT / "wfcrl-benchmark"
assert (ENV_REPO / "wfcrl" / "simple_env.py").exists()
assert (BENCH / "algos" / "baseline_ippo.py").exists()
print("Python:", sys.executable)
print("WFCRL:", ENV_REPO)
print("Algorithms:", BENCH)

## Environment

The original decentralized observation and yaw action remain unchanged. `log=False` keeps a quick inspection run from accumulating a long history.

In [ ]:
sys.path.insert(0, str(ENV_REPO))
from wfcrl import environments as envs

env = envs.make("Dec_Ablaincourt_Floris", log=False, max_num_steps=151, load_coef=0.1)
env.reset(options={"wind_speed": 8.0, "wind_direction": 270.0})
agent = env.possible_agents[0]
print("agents:", env.possible_agents)
print("observation:", env.observe(agent))
print("action space:", env.action_space(agent))
env.close()

In [ ]:
sys.path.insert(0, str(BENCH / "algos"))
from extractors import VectorExtractor
from baseline_ippo import Agent as IPPOAgent
from baseline_mappo import Agent as MAPPOAgent, SharedCritic

local_obs = VectorExtractor(env.observation_space(agent)).space
local_action = VectorExtractor(env.action_space(agent)).space
global_obs = VectorExtractor(env.state_space).space
print("IPPO independent actor-critic:", IPPOAgent(local_obs, local_action, (64, 64)))
print("MAPPO local actor:", MAPPOAgent(local_obs, local_action, (64, 64)))
print("MAPPO shared global critic:", SharedCritic(global_obs, (64, 64)))

## Train the authors' baselines

Set `ALGORITHMS` to one or both methods. The scripts retain their own PPO settings. `--no-track` disables the optional Weights & Biases login; TensorBoard logging stays on. `TOTAL_TIMESTEPS` is per training run. Training both methods at the authors' scale takes substantial simulator time.

For comparison with EnvPG, the command below sets `episode_length=151`: WFCRL consumes one simulator update during reset, leaving 150 controlled steps. This differs from the authors' default of 150. Use 150 to reproduce that default exactly.

In [ ]:
ALGORITHMS = ("ippo", "mappo")
SEEDS = (1,)
SCENARIO = "windrose"
LOAD_COEF = 0.1  # Wind-MARL comparison; use 1.0 for the paper setting
TOTAL_TIMESTEPS = 100_000
EPISODE_LENGTH = 151  # 150 controlled steps after WFCRL reset

run_env = os.environ.copy()
run_env["MPLBACKEND"] = "Agg"
run_env["PYTHONPATH"] = str(ENV_REPO) + os.pathsep + run_env.get("PYTHONPATH", "")
run_env["MPLCONFIGDIR"] = str(ROOT / "__simul__" / "matplotlib")
Path(run_env["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

def train_baseline(algorithm, seed):
    assert algorithm in {"ippo", "mappo"}
    command = [sys.executable, f"algos/baseline_{algorithm}.py",
               "--env-id", "Dec_Ablaincourt_Floris", "--scenario", SCENARIO,
               "--wind-data", "data/smarteole.csv", "--seed", str(seed),
               "--load-coef", str(LOAD_COEF), "--episode-length", str(EPISODE_LENGTH),
               "--total-timesteps", str(TOTAL_TIMESTEPS),
               "--no-track", "--no-cuda"]
    print("Running:", " ".join(command), flush=True)
    subprocess.run(command, cwd=BENCH, env=run_env, check=True)

for algorithm in ALGORITHMS:
    for seed in SEEDS:
        train_baseline(algorithm, seed)

## TensorBoard

Run the following cell to view the authors' training and evaluation curves. For a direct EnvPG comparison, use the same wind cases, horizon, reward coefficient and simulator-call budget; this notebook does not change the authors' policy observations.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {BENCH / "runs"}